In [3]:
import pandas as pd
import warnings
pd.set_option('display.float_format', lambda x: '%.3f' % x)
warnings.filterwarnings("ignore")

In [4]:
app_train_fe=pd.read_parquet('data/interim/app_train_fe.parquet')
app_test_fe=pd.read_parquet('data/interim/app_test_fe.parquet')
bureau_agg=pd.read_parquet('data/interim/bureau_agg1.parquet')
bureau_active=pd.read_parquet('data/interim/bureau_agg_active.parquet')
bureau_closed=pd.read_parquet('data/interim/bureau_agg_closed.parquet')

FileNotFoundError: [Errno 2] No such file or directory: 'data/interim/app_train_fe.parquet'

In [ ]:
final_train_matrix=app_train_fe.merge(bureau_agg,on='SK_ID_CURR',how='left')
final_train_matrix=final_train_matrix.merge(bureau_active,on='SK_ID_CURR',how='left')
final_train_matrix=final_train_matrix.merge(bureau_closed,on='SK_ID_CURR',how='left')

In [ ]:
final_test_matrix=app_test_fe.merge(bureau_agg,on='SK_ID_CURR',how='left')
final_test_matrix=final_test_matrix.merge(bureau_active,on='SK_ID_CURR',how='left')
final_test_matrix=final_test_matrix.merge(bureau_closed,on='SK_ID_CURR',how='left')

In [ ]:
final_train_matrix.shape

(307511, 1015)

In [ ]:
final_test_matrix.shape

(48744, 1014)

In [ ]:

missing_pct = final_train_matrix.isnull().mean().sort_values(ascending=False)
missing_pct.head(30)

ACTIVE_BUREAU_B_TENURE_GAP_STD               1.000
ACTIVE_BUREAU_B_ACTUAL_TENURE_STD            1.000
ACTIVE_BUREAU_DAYS_ENDDATE_FACT_STD          1.000
CLOSED_BUREAU_B_MAX_OVERDUE_TO_DEBT_STD      0.998
CLOSED_BUREAU_B_CURR_OVERDUE_TO_DEBT_STD     0.997
CLOSED_BUREAU_B_OVERDUE_TO_DEBT_STD          0.997
ACTIVE_BUREAU_B_TENURE_GAP_MAX               0.995
ACTIVE_BUREAU_B_TENURE_GAP_MIN               0.995
ACTIVE_BUREAU_B_TENURE_GAP_MEAN              0.995
ACTIVE_BUREAU_B_ACTUAL_TENURE_MEAN           0.995
ACTIVE_BUREAU_B_ACTUAL_TENURE_MIN            0.995
ACTIVE_BUREAU_DAYS_ENDDATE_FACT_MEAN         0.995
ACTIVE_BUREAU_DAYS_ENDDATE_FACT_MAX          0.995
ACTIVE_BUREAU_DAYS_ENDDATE_FACT_MIN          0.995
ACTIVE_BUREAU_B_ACTUAL_TENURE_MAX            0.995
CLOSED_BUREAU_B_MAX_OVERDUE_TO_DEBT_MEAN     0.993
CLOSED_BUREAU_B_MAX_OVERDUE_TO_DEBT_MAX      0.993
ACTIVE_BUREAU_B_CURRENT_TO_MAX_OVERDUE_STD   0.984
ACTIVE_BUREAU_B_OVERDUE_TO_MAX_OVERDUE_STD   0.984
CLOSED_BUREAU_B_CURR_OVERDUE_TO

In [ ]:
# dropping the columns with > 99.5% missing values
drop_columns=[
'ACTIVE_BUREAU_B_TENURE_GAP_STD'          ,  
'ACTIVE_BUREAU_B_ACTUAL_TENURE_STD'        ,    
'ACTIVE_BUREAU_DAYS_ENDDATE_FACT_STD'       ,
'CLOSED_BUREAU_B_MAX_OVERDUE_TO_DEBT_STD'    , 
'CLOSED_BUREAU_B_CURR_OVERDUE_TO_DEBT_STD'    , 
'CLOSED_BUREAU_B_OVERDUE_TO_DEBT_STD'    ,
'ACTIVE_BUREAU_B_TENURE_GAP_MAX'        ,
'ACTIVE_BUREAU_B_TENURE_GAP_MIN'              ,
'ACTIVE_BUREAU_B_TENURE_GAP_MEAN'             ,
'ACTIVE_BUREAU_B_ACTUAL_TENURE_MEAN'          ,
'ACTIVE_BUREAU_B_ACTUAL_TENURE_MIN'           ,
'ACTIVE_BUREAU_DAYS_ENDDATE_FACT_MEAN'        ,
'ACTIVE_BUREAU_DAYS_ENDDATE_FACT_MAX'         ,
'ACTIVE_BUREAU_DAYS_ENDDATE_FACT_MIN'         ,
'ACTIVE_BUREAU_B_ACTUAL_TENURE_MAX'
]
final_train_matrix=final_train_matrix.drop(drop_columns,axis=1)
final_test_matrix=final_test_matrix.drop(drop_columns,axis=1)

We trained the baseline LightGBM and the auc was 0.77. To further improve, we created a new dataset with debt type.

In [ ]:
bureau_debt_type=pd.read_parquet('data/interim/bureau_debt_type.parquet')

In [ ]:
bureau_debt_type.reset_index()
bureau_debt_type

,total_consumer_credit_debt,total_consumer_credit_count,total_consumer_credit_overdue,total_consumer_credit_max_overdue,consumer_credit_overdue_to_typedebt,consumer_credit_max_overdue_to_typedebt,consumer_credit_overdue_to_total,consumer_credit_max_overdue_to_total,consumer_credit_debt_share,total_credit_card_debt,...,mortgage_debt_share,total_microloan_debt,total_microloan_count,total_microloan_overdue,total_microloan_max_overdue,microloan_overdue_to_typedebt,microloan_max_overdue_to_typedebt,microloan_overdue_to_total,microloan_max_overdue_to_total,microloan_debt_share
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
215354,4394913.300,7.000,0.000,77674.500,0.000,0.018,0.000,0.018,1.000,495000.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162297,854886.150,3.000,0.000,14985.000,0.000,0.018,0.000,0.018,1.000,342000.000,...,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
402440,89910.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
238881,965739.060,5.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,319500.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
222183,3321567.000,4.000,0.000,20803.770,0.000,0.006,0.000,0.006,1.000,127840.500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207190,450000.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
324956,19800.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
448157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
final_train_matrix=final_train_matrix.merge(bureau_debt_type,on='SK_ID_CURR',how='left')
final_test_matrix=final_test_matrix.merge(bureau_debt_type,on='SK_ID_CURR',how='left')

In [ ]:
app_test_ids=final_test_matrix['SK_ID_CURR']
app_test_ids.to_csv('data/interim/app_test_ids.csv')

In [ ]:
final_train_matrix.to_parquet('data/interim/final_train.parquet')
final_test_matrix.to_parquet('data/interim/final_test.parquet')

In [ ]:
final_train_matrix.shape

(307511, 1045)